# Analyse der Klima-Komposita auf deutschsprachigen Titelseiten

**Forschungsfrage**: Wie hat sich die Verwendung der häufigsten Begriffe mit dem Wortstamm 'Klima' auf deutschsprachigen Online-Titelseiten zwischen 2021 und 2025 entwickelt?

**Zeitraum**: 21.04.2022 bis 08.02.2025 (Scraper-Cutoff bis letztes gecrawltes Datum; Update 20.03.2026)

**Drei Begriffe (finale Auswahl)**:
- Klimawandel: wandel, wandels
- Klimakrise: krise, krisen
- Klimaschutz: schutz, schutzes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys

# Einheitliche Pfadlogik wie in 01_lake_to_dwh
notebook_dir = (
    os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
)
sys.path.append(os.path.join(notebook_dir, "..", "pylib"))
from handle_sqlite import read_table_as_dataframe
db_path = os.path.join(notebook_dir, "..", "data_output", "dwh_data.db")


In [ ]:
# Design-Einstellungen: Schwarz-weiß
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['figure.facecolor'] = 'white'

# Ausgabepfad
plot_dir = os.path.join(notebook_dir, "..", "data_output", "plots")
os.makedirs(plot_dir, exist_ok=True)

In [ ]:

# Daten laden (processed-Tabellen aus SQLite)
df_context = read_table_as_dataframe('context_processed', db_path)
df_meta = read_table_as_dataframe('newspapers', db_path)

df_meta['data_published'] = pd.to_datetime(df_meta['data_published'])

print(f"Context-Daten: {len(df_context)} Zeilen")
print(f"Meta-Daten: {len(df_meta)} Zeilen")

In [ ]:
SCRAPER_CUTOFF = pd.Timestamp('2022-04-21')
END_DATE = pd.Timestamp('2026-01-31')  # z. B. '2025-01-31' oder None

df_meta_filtered = df_meta[df_meta['data_published'] >= SCRAPER_CUTOFF].copy()

if END_DATE:
    df_meta_filtered = df_meta_filtered[df_meta_filtered['data_published'] <= END_DATE].copy()

print(f"Einträge insgesamt:                                    {len(df_meta)} Nennungen")
print(f"Daten von {SCRAPER_CUTOFF} bis {END_DATE}: {len(df_meta_filtered)} Nennungen")

In [ ]:
# show cut
daily = df_meta.groupby('data_published').size()

fig, ax = plt.subplots(figsize=(14, 4))

# Höherer Kontrast: sehr helle Außenbereiche, dunkler Analysezeitraum
outside_color = '#E8E8E8'
inside_color = '#111111'
colors = [inside_color if SCRAPER_CUTOFF <= pd.to_datetime(d) <= END_DATE else outside_color for d in daily.index]

ax.bar(daily.index, daily.values, color=colors, edgecolor='white', linewidth=0.35, width=1.0)
ax.axvspan(SCRAPER_CUTOFF, END_DATE, color='#000000', alpha=0.06, label='Analysezeitraum')

ax.axvline(SCRAPER_CUTOFF, color='black', linestyle='--', linewidth=1.2, label=f'Cutoff {SCRAPER_CUTOFF.date()}')
ax.axvline(END_DATE, color='black', linestyle=':', linewidth=1.2, label=f'Ende {END_DATE.date()}')

ax.set_facecolor('white')
ax.set_xlabel('Datum')
ax.set_ylabel('Einträge')
ax.set_title('Tagesverlauf der Erhebung mit Analysezeitraum')
ax.grid(axis='y', alpha=0.15)
ax.legend(fontsize=9, frameon=False, ncol=3, loc='upper left')

for axis in (ax,):
    axis.spines['top'].set_visible(False)
    axis.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Merge: meta is left → all meta rows kept, context joined where available
df = df_meta_filtered[['newspaper_id', 'data_published']].merge(df_context, on='newspaper_id', how='left')

print(f"Zeitraum: {df['data_published'].min()} bis {df['data_published'].max()}")
print(f"Einträge insgesamt: {len(df_meta)} Nennungen")
print(f"Daten von {SCRAPER_CUTOFF} bis {END_DATE}: {len(df)} Nennungen")

In [ ]:
# Finale Mapping-Definition für die Hauptanalyse, nach dem processing:
# Klimawandel: wandel, Klimakrise: krise, Klimaschutz: schutz
SUFFIX_TO_BEGRIFF = {
    'wandel': 'Klimawandel',
    'krise':  'Klimakrise',
    'schutz': 'Klimaschutz',
}

df['begriff'] = (
    df['suffix_lemma']
    .str.lower()
    .str.strip()
    .map(SUFFIX_TO_BEGRIFF)
)

print("\nVerwendete finale Suffixe:")
for suffix, begriff in SUFFIX_TO_BEGRIFF.items():
    print(f"- {begriff}: {suffix}")

df_drei = df[df['begriff'].notna()].copy()
print(f"Nennungen der drei Begriffe (finale Auswahl): {len(df_drei)}")
print(df_drei['begriff'].value_counts())

## Analyse 1: Gesamtverteilung der Begriffe

In [ ]:
# Absolute Häufigkeiten
counts = df_drei['begriff'].value_counts()
total = counts.sum()
percentages = (counts / total * 100).round(2)

fig, ax = plt.subplots(figsize=(7, 4))

# Zurückhaltende Graustufen für die drei Begriffe
bar_colors = ['#1A1A1A', '#7A7A7A', '#C9C9C9']
bars = ax.bar(counts.index, counts.values, color=bar_colors, edgecolor='white', linewidth=0.6)

# Prozentzahl über jeden Balken
for bar, pct in zip(bars, percentages.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(counts.max() * 0.02, 3),
        f'{pct}%',
        ha='center',
        va='bottom',
        fontsize=10,
        color='black'
    )

ax.set_title('Gesamtverteilung der Begriffe (21.04.2022 – 08.02.2025)')
ax.set_ylabel('Absolute Häufigkeit')
ax.set_xlabel('Begriff')
ax.set_ylim(0, counts.max() * 1.18)  # Platz für Labels
ax.grid(axis='y', alpha=0.2)

for axis in (ax,):
    axis.spines['top'].set_visible(False)
    axis.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## Analyse 2: Zeitliche Entwicklung (Quartalweise)

In [ ]:
# Quartal berechnen
df_drei['quarter'] = df_drei['data_published'].dt.to_period('Q')
# Gruppieren: Quartal x Begriff
quarterly_counts = df_drei.groupby(['quarter', 'begriff']).size().unstack(fill_value=0)
# Relative Anteile pro Quartal berechnen
quarterly_total = quarterly_counts.sum(axis=1)
quarterly_pct = quarterly_counts.div(quarterly_total, axis=0) * 100
print("\n=== ANALYSE 2: Relative Anteile pro Quartal (%) ===")
print(quarterly_pct.round(2).to_string())


In [ ]:
# Auch absolute Zahlen pro Quartal zeigen
print("\n=== ANALYSE 2b: Absolute Häufigkeiten pro Quartal ===")
quarterly_counts['Gesamt'] = quarterly_total
print(quarterly_counts.to_string())

## Analyse 3: Neutral vs. Alarmistisch

In [ ]:
# Kategorien:
# Klimawandel = neutral
# Klimakrise = alarmistisch
# Klimaschutz = handlungsorientiert (separat oder ausklammern)

# Nur Klimawandel und Klimakrise für diese Analyse
df_neutral_alarm = df_drei[df_drei['begriff'].isin(['Klimawandel', 'Klimakrise'])].copy()

def map_kategorie(begriff):
    if begriff == 'Klimawandel':
        return 'Neutral'
    elif begriff == 'Klimakrise':
        return 'Alarmistisch'
    return None

df_neutral_alarm['kategorie'] = df_neutral_alarm['begriff'].apply(map_kategorie)

# Quartalweise gruppieren
neutral_alarm_counts = df_neutral_alarm.groupby(['quarter', 'kategorie']).size().unstack(fill_value=0)
neutral_alarm_total = neutral_alarm_counts.sum(axis=1)
neutral_alarm_pct = neutral_alarm_counts.div(neutral_alarm_total, axis=0) * 100

print("\n=== ANALYSE 3: Neutral vs. Alarmistisch pro Quartal (%) ===")
print(neutral_alarm_pct.round(2).to_string())

---
## Grafiken

### Grafik 1: Zeitliche Entwicklung der drei Begriffe (Liniendiagramm)

In [ ]:
import pandas as pd
import os
import sys


# Einheitliche Pfadlogik wie im Notebook
notebook_dir = (
    os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
)
sys.path.append(os.path.join(notebook_dir, "..", "pylib"))
from handle_sqlite import read_table_as_dataframe

db_path = os.path.join(notebook_dir, "..", "data_output", "dwh_data.db")

# Daten laden
df_context = read_table_as_dataframe("context_processed", db_path)
df_meta = read_table_as_dataframe("newspapers", db_path)

# Merge + Datumsfilter wie im Notebook
df = df_context.merge(df_meta[["newspaper_id", "data_published"]], on="newspaper_id", how="left")
df["date"] = pd.to_datetime(df["data_published"])

SCRAPER_CUTOFF = "2022-04-21"
END_DATE = "2025-02-08"
df = df[(df["date"] >= SCRAPER_CUTOFF) & (df["date"] <= END_DATE)].copy()

# Finale Begriffsmappung wie im Notebook
df["suffix_clean"] = df["suffix"].str.lower().str.strip()
FINAL_MAPPING = {
    "Klimawandel": ["wandel", "wandels"],
    "Klimakrise": ["krise", "krisen"],
    "Klimaschutz": ["schutz", "schutzes"],
}

def map_to_begriff(suffix):
    if pd.isna(suffix):
        return None
    suffix = str(suffix).lower().strip()
    for begriff, suffixe in FINAL_MAPPING.items():
        if suffix in suffixe:
            return begriff
    return None

df["begriff"] = df["suffix_clean"].apply(map_to_begriff)
df_drei = df[df["begriff"].notna()].copy()

# Monatsskala statt Quartal + absolute Häufigkeiten
df_drei["month"] = df_drei["date"].dt.to_period("M")
monthly_counts = (
    df_drei.groupby(["month", "begriff"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Klimaschutz", "Klimawandel", "Klimakrise"], fill_value=0)
)

# Monats-Gesamtsumme
monthly_counts["Gesamt"] = monthly_counts.sum(axis=1)

# Klartextausgabe
print(f"Zeitraum (min/max): {df['date'].min()} bis {df['date'].max()}")
print(f"Anzahl Nennungen in df_drei: {len(df_drei)}")
print("\nNennungen je Begriff:")
print(df_drei["begriff"].value_counts().to_string())
print("\nMonatstabelle (absolute Häufigkeiten):")
print(monthly_counts.to_string())
print("\nKopf der Monatstabelle:")
print(monthly_counts.head(10).to_string())
print("\nLetzter Monat:")
print(monthly_counts.tail(1).to_string())

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))

x = range(len(monthly_counts))
labels = [str(m) for m in monthly_counts.index]

# Drei Begriffe als Linien (absolute Häufigkeiten pro Monat)
ax.plot(x, monthly_counts["Klimaschutz"], linestyle="-", marker="o", color="black", linewidth=1.8, markersize=4, label="Klimaschutz")
ax.plot(x, monthly_counts["Klimawandel"], linestyle="--", marker="s", color="dimgray", linewidth=1.8, markersize=4, label="Klimawandel")
ax.plot(x, monthly_counts["Klimakrise"], linestyle=":", marker="^", color="gray", linewidth=1.8, markersize=4, label="Klimakrise")

# Optional: Gesamt als Referenz (auskommentieren, wenn nicht gewünscht)
# ax.plot(x, monthly_counts["Gesamt"], linestyle="-.", color="lightgray", linewidth=1.5, label="Gesamt")

ax.set_xticks(x[::2])  # jedes 2. Monatslabel für bessere Lesbarkeit
ax.set_xticklabels(labels[::2], rotation=45, ha="right")

ax.set_xlabel("Monat")
ax.set_ylabel("Absolute Häufigkeit")
ax.set_title("Monatliche absolute Häufigkeiten der Klima-Begriffe (2022–2025)")
ax.grid(True, alpha=0.3)
ax.legend(loc="best", frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
# Daten vorbereiten
quarterly_pct_plot = quarterly_pct.drop(columns=['Gesamt'], errors='ignore')

fig, ax = plt.subplots(figsize=(10, 6))

# Linienarten definieren (schwarz-weiß tauglich)
linestyles = {'Klimaschutz': '-', 'Klimawandel': '--', 'Klimakrise': ':'}
markers = {'Klimaschutz': 'o', 'Klimawandel': 's', 'Klimakrise': '^'}

for begriff in ['Klimaschutz', 'Klimawandel', 'Klimakrise']:
    if begriff in quarterly_pct_plot.columns:
        ax.plot(
            range(len(quarterly_pct_plot)),
            quarterly_pct_plot[begriff],
            linestyle=linestyles[begriff],
            marker=markers[begriff],
            color='black',
            linewidth=1.5,
            markersize=5,
            label=begriff
        )

# X-Achsen-Labels
ax.set_xticks(range(len(quarterly_pct_plot)))
ax.set_xticklabels([str(q) for q in quarterly_pct_plot.index], rotation=45, ha='right')

ax.set_xlabel('Quartal')
ax.set_ylabel('Relativer Anteil (%)')
ax.set_title('Entwicklung der Klima-Begriffe auf deutschsprachigen Titelseiten (2022–2025)')
ax.legend(loc='best', frameon=True)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(plot_dir, 'grafik_1_zeitliche_entwicklung.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Grafik 1 gespeichert: {os.path.join(plot_dir, 'grafik_1_zeitliche_entwicklung.png')}")

### Grafik 2a: Neutral vs. Alarmistisch (Flächendiagramm)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Flächendiagramm mit Grautönen
x = range(len(neutral_alarm_pct))

ax.fill_between(x, 0, neutral_alarm_pct['Neutral'],
                color='lightgray', label='Neutral (Klimawandel)', alpha=0.8)
ax.fill_between(x, neutral_alarm_pct['Neutral'], 100,
                color='darkgray', label='Alarmistisch (Klimakrise)', alpha=0.8)

# X-Achsen-Labels
ax.set_xticks(x)
ax.set_xticklabels([str(q) for q in neutral_alarm_pct.index], rotation=45, ha='right')

ax.set_xlabel('Quartal')
ax.set_ylabel('Anteil (%)')
ax.set_title('Anteil neutraler vs. alarmistischer Begriffe im Zeitverlauf')
ax.legend(loc='upper right', frameon=True)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.savefig(os.path.join(plot_dir, 'grafik_2a_neutral_vs_alarmistisch_flaeche.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Grafik 2a gespeichert: {os.path.join(plot_dir, 'grafik_2a_neutral_vs_alarmistisch_flaeche.png')}")

### Grafik 2b: Neutral vs. Alarmistisch (Liniendiagramm)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

x = range(len(neutral_alarm_pct))

ax.plot(x, neutral_alarm_pct['Neutral'],
        linestyle='-', marker='o', color='black', linewidth=2,
        label='Neutral (Klimawandel)')
ax.plot(x, neutral_alarm_pct['Alarmistisch'],
        linestyle='--', marker='s', color='gray', linewidth=2,
        label='Alarmistisch (Klimakrise)')

# X-Achsen-Labels
ax.set_xticks(x)
ax.set_xticklabels([str(q) for q in neutral_alarm_pct.index], rotation=45, ha='right')

ax.set_xlabel('Quartal')
ax.set_ylabel('Anteil (%)')
ax.set_title('Anteil neutraler vs. alarmistischer Begriffe im Zeitverlauf')
ax.legend(loc='best', frameon=True)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(plot_dir, 'grafik_2b_neutral_vs_alarmistisch_linien.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Grafik 2b gespeichert: {os.path.join(plot_dir, 'grafik_2b_neutral_vs_alarmistisch_linien.png')}")

### Grafik 3: Absolute Häufigkeiten (Balkendiagramm)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Daten
begriffe = ['Klimaschutz', 'Klimawandel', 'Klimakrise']
counts_sorted = [counts.get(b, 0) for b in begriffe]

# Balken mit Grautönen
colors = ['black', 'gray', 'lightgray']
bars = ax.bar(begriffe, counts_sorted, color=colors, edgecolor='black')

# Werte über Balken anzeigen
for bar, count in zip(bars, counts_sorted):
    ax.annotate(f'{count:,}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 5), textcoords='offset points',
                ha='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Begriff')
ax.set_ylabel('Anzahl Nennungen')
ax.set_title('Absolute Häufigkeit der Klima-Begriffe (2022–2025)')
ax.set_ylim(0, max(counts_sorted) * 1.15)

plt.tight_layout()
plt.savefig(os.path.join(plot_dir, 'grafik_3_absolute_haeufigkeiten.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Grafik 3 gespeichert: {os.path.join(plot_dir, 'grafik_3_absolute_haeufigkeiten.png')}")

---
## Erkenntnisse

Die konkreten Zahlen werden nach Ausführung des Notebooks hier zusammengefasst.

In [ ]:
# Zusammenfassung der wichtigsten Zahlen
print("\n" + "="*60)
print("ZUSAMMENFASSUNG DER ERGEBNISSE")
print("="*60)

print(f"\n📊 GESAMTZAHLEN ({SCRAPER_CUTOFF} bis {END_DATE})")
print(f"   Gesamtzahl Nennungen: {total:,}")
for b in begriffe:
    c = counts.get(b, 0)
    p = (c / total * 100)
    print(f"   {b}: {c:,} ({p:.1f}%)")

print(f"\n📈 ZEITLICHE ENTWICKLUNG (erstes vs. letztes Quartal)")
first_q = quarterly_pct_plot.iloc[0]
last_q = quarterly_pct_plot.iloc[-1]
for b in begriffe:
    if b in first_q.index:
        diff = last_q[b] - first_q[b]
        trend = '↑' if diff > 0 else '↓' if diff < 0 else '→'
        print(f"   {b}: {first_q[b]:.1f}% → {last_q[b]:.1f}% ({trend} {abs(diff):.1f} Pp.)")

print(f"\n⚖️ NEUTRAL VS. ALARMISTISCH")
first_na = neutral_alarm_pct.iloc[0]
last_na = neutral_alarm_pct.iloc[-1]
print(f"   Erstes Quartal: Neutral {first_na['Neutral']:.1f}% | Alarmistisch {first_na['Alarmistisch']:.1f}%")
print(f"   Letztes Quartal: Neutral {last_na['Neutral']:.1f}% | Alarmistisch {last_na['Alarmistisch']:.1f}%")

print("\n" + "="*60)